# statistiques from scratch

reference : joel grus, "data science from scratch", chapitre 5

toutes les fonctions sont implementees sans numpy, pandas ou sklearn - uniquement avec des listes python.

In [22]:
# imports necessaires
import math
import csv
from pathlib import Path

In [23]:
def mean(xs: list[float]) -> float:
    """donne la moyenne d'une liste de chiffres."""
    # on additionne tout et on divise par le nombre d'elements
    return sum(xs) / len(xs)


def median(xs: list[float]) -> float:
    """trouve la valeur qui coupe la liste en deux."""
    # d'abord on trie la liste pour y voir plus clair
    n = len(xs)
    sorted_xs = sorted(xs)
    midpoint = n // 2

    if n % 2 == 1:
        # si c'est impair, on prend pile l'element du milieu
        return sorted_xs[midpoint]
    else:
        # si c'est pair, on fait la moyenne des deux chiffres du centre
        lo = midpoint - 1
        hi = midpoint
        return (sorted_xs[lo] + sorted_xs[hi]) / 2


def variance(xs: list[float]) -> float:
    """calcule la variance pour voir comment les chiffres s'etalent."""
    # on mesure l'ecart moyen par rapport a la moyenne
    n = len(xs)
    if n < 2:
        return 0.0
    mu = mean(xs)
    # on fait la somme des ecarts au carre et on divise
    return sum((x - mu) ** 2 for x in xs) / (n - 1)


def standard_deviation(xs: list[float]) -> float:
    """donne l'ecart-type, c'est plus facile a lire que la variance."""
    # c'est simplement la racine carree de la variance
    return math.sqrt(variance(xs))


# test rapide
test_data = [1, 2, 3, 4, 5]
print(f"test sur {test_data}")
print(f"moyenne : {mean(test_data)}")
print(f"mediane : {median(test_data)}")
print(f"variance : {variance(test_data):.2f}")
print(f"ecart-type : {standard_deviation(test_data):.2f}")

test sur [1, 2, 3, 4, 5]
moyenne : 3.0
mediane : 3
variance : 2.50
ecart-type : 1.58


In [24]:
def covariance(xs: list[float], ys: list[float]) -> float:
    """regarde si deux series de chiffres ont tendance a varier ensemble."""
    # on verifie si x et y grimpent ou descendent en meme temps
    n = len(xs)
    if n < 2:
        return 0.0
    mu_x = mean(xs)
    mu_y = mean(ys)
    return sum((x - mu_x) * (y - mu_y) for x, y in zip(xs, ys)) / (n - 1)


def correlation(xs: list[float], ys: list[float]) -> float:
    """
    calcule le coefficient de pearson (entre -1 et 1).
    mesure la force de la relation lineaire entre deux variables.
    """
    # c'est le rapport entre la covariance et les ecarts-types
    stdev_x = standard_deviation(xs)
    stdev_y = standard_deviation(ys)
    if stdev_x > 0 and stdev_y > 0:
        return covariance(xs, ys) / (stdev_x * stdev_y)
    else:
        # si l'une des listes est toute plate, la correlation est nulle
        return 0


# test rapide
x_test = [1, 2, 3, 4, 5]
y_test = [2, 4, 6, 8, 10]  # correlation parfaite
print(f"test correlation entre {x_test} et {y_test}")
print(f"covariance : {covariance(x_test, y_test):.2f}")
print(f"correlation : {correlation(x_test, y_test):.4f}")

test correlation entre [1, 2, 3, 4, 5] et [2, 4, 6, 8, 10]
covariance : 5.00
correlation : 1.0000


## test sur donnees reelles dvf toulon

chargement des donnees nettoyees et calcul des statistiques descriptives

In [ ]:
# chargement des donnees dvf nettoyees
surfaces = []
prix = []

csv_path = Path.cwd().parent / "data" / "DVF-83-Toulon-2024-2025advanced.csv"

with open(csv_path, mode="r", encoding="utf-8") as f:
    reader = csv.DictReader(f, delimiter=";")
    for row in reader:
        try:
            s = float(row["surface_totale"])
            p = float(row["valeur_fonciere"])
            surfaces.append(s)
            prix.append(p)
        except (ValueError, TypeError, KeyError):
            continue

print(f"{len(surfaces)} transactions chargees")
print(f"\nplage de surfaces : {min(surfaces):.0f} - {max(surfaces):.0f} m²")
print(f"plage de prix : {min(prix):,.0f} - {max(prix):,.0f} euros")

In [26]:
# statistiques descriptives sur les surfaces
print("=== statistiques sur les surfaces (m²) ===")
print(f"moyenne : {mean(surfaces):.2f} m²")
print(f"mediane : {median(surfaces):.2f} m²")
print(f"ecart-type : {standard_deviation(surfaces):.2f} m²")
print(f"variance : {variance(surfaces):.2f}")
print()

# statistiques descriptives sur les prix
print("=== statistiques sur les prix (euros) ===")
print(f"moyenne : {mean(prix):,.2f} euros")
print(f"mediane : {median(prix):,.2f} euros")
print(f"ecart-type : {standard_deviation(prix):,.2f} euros")
print(f"variance : {variance(prix):,.2f}")
print()

# prix au m²
prix_m2 = [p / s for p, s in zip(prix, surfaces)]
print("=== statistiques sur le prix au m² (euros/m²) ===")
print(f"moyenne : {mean(prix_m2):,.2f} euros/m²")
print(f"mediane : {median(prix_m2):,.2f} euros/m²")
print(f"ecart-type : {standard_deviation(prix_m2):,.2f} euros/m²")

=== statistiques sur les surfaces (m²) ===
moyenne : 60.39 m²
mediane : 60.00 m²
ecart-type : 23.02 m²
variance : 530.07

=== statistiques sur les prix (euros) ===
moyenne : 170,226.67 euros
mediane : 146,000.00 euros
ecart-type : 95,314.48 euros
variance : 9,084,849,524.81

=== statistiques sur le prix au m² (euros/m²) ===
moyenne : 2,859.21 euros/m²
mediane : 2,682.04 euros/m²
ecart-type : 1,117.52 euros/m²


In [27]:
# relation entre surface et prix
print("=== relation surface <-> prix ===")
print(f"covariance : {covariance(surfaces, prix):,.2f}")
print(f"correlation de pearson : {correlation(surfaces, prix):.4f}")
print()

# interpretation
corr_val = correlation(surfaces, prix)
if corr_val > 0.7:
    interpretation = "correlation tres forte"
elif corr_val > 0.5:
    interpretation = "correlation forte"
elif corr_val > 0.3:
    interpretation = "correlation moderee"
else:
    interpretation = "correlation faible"

print(f"interpretation : {interpretation}")
print(f"cela signifie que la surface explique une partie significative du prix")

=== relation surface <-> prix ===
covariance : 1,421,901.55
correlation de pearson : 0.6480

interpretation : correlation forte
cela signifie que la surface explique une partie significative du prix
